# GOLD LAYER

# Various Imports and Python Environment Configuration. Create the spark session.

In [1]:
import os
import sys
from pathlib import Path

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SILVER_DIR = PROJECT_ROOT / "delta" / "silver"
GOLD_DIR = PROJECT_ROOT / "delta" / "gold"

GOLD_DIR.mkdir(parents=True, exist_ok=True)

builder = (
    SparkSession.builder
    .appName("gold-analytics")
    .master("local[*]")
    .config(
        "spark.sql.extensions",
        "io.delta.sql.DeltaSparkSessionExtension",
    )
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog",
    )
    .config("spark.sql.shuffle.partitions", "8")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("WARN")

print("Python      :", sys.executable)
print("Silver      :", SILVER_DIR)
print("Gold        :", GOLD_DIR)


Python      : C:\lufthansa-de-exercise\.venv\Scripts\python.exe
Silver      : C:\lufthansa-de-exercise\delta\silver
Gold        : C:\lufthansa-de-exercise\delta\gold


# Simple checking if every table is present for the gold layer.

In [2]:
REQUIRED_SILVER_TABLES = [
    "order_enriched",
    "customers",
]

missing_tables = [
    table_name
    for table_name in REQUIRED_SILVER_TABLES
    if not (SILVER_DIR / table_name / "_delta_log").exists()
]

if missing_tables:
    raise FileNotFoundError(
        "Missing silver delta tables: " + ", ".join(missing_tables)
    )

print("All required slver delta tables are available.")

All required slver delta tables are available.


# Create functions to read the silver table and write in the gold tables.

In [3]:
PARTITION_COLS = ["year", "month", "day"]
write_results = {}

def read_silver(table_name):
    # load the silver table
    return (
        spark.read
        .format("delta")
        .load(str(SILVER_DIR / table_name))
    )

def write_gold(df, table_name):
    # validate the partitioned table and write it later
    missing = [
        column
        for column in PARTITION_COLS
        if column not in df.columns
    ]
    if missing:
        raise ValueError(
            f"{table_name} is missing partition columns: {missing}"
        )

    output_path = GOLD_DIR / table_name
    
    row_count = df.count()

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .partitionBy(*PARTITION_COLS)
        .save(str(output_path))
    )

    write_results[table_name] = {
        "path": str(output_path),
        "expected_count": row_count,
    }

    print(f"{table_name:<38} rows={row_count:>8,}")
    return df

# Load the enriched table created in silver ...

The table has one row per order item. The views needed to be created have these grains:
- Cumulative Sales per Customer - one row per order
- Rolling Average Delivery Time per Product Category - one row per order and category
- KPI Summary Table: - one row per order and seller.

In [4]:
# we read the enriched table
enriched = read_silver("order_enriched")

#read the customer and select 2 columns only
customers = (
    read_silver("customers")
    .select("customer_id", "customer_unique_id")
    .dropDuplicates(["customer_id"])
)

# read table from enriched but group by order while aggregating price, profit and count
order_totals = (
    enriched
    .groupBy(
        "order_id",
        "customer_id",
        "order_status",
        "order_purchase_timestamp",
        "delivery_time_days",
        "year",
        "month",
        "day",
    )
    .agg(
        F.round(F.sum("total_price"), 2).alias("total_price"),
        F.round(F.sum("profit_margin"), 2).alias("total_profit"),
        F.count("*").alias("item_count"),
    )
    .join(
        F.broadcast(customers),
        on="customer_id",
        how="left",
    )
)
# same thing here in term of process, but we have one row per order and category
order_category = (
    enriched
    .groupBy(
        "order_id",
        "product_category_name",
        "order_purchase_timestamp",
        "delivery_time_days",
        "year",
        "month",
        "day",
    )
    .agg(
        F.round(F.sum("total_price"), 2).alias(
            "category_order_sales"
        )
    )
)

# same thing here in term of process, but we have one row per order and seller
order_seller = (
    enriched
    .groupBy(
        "order_id",
        "seller_id",
        "seller_state",
        "delivery_time_days",
        "year",
        "month",
        "day",
    )
    .agg(
        F.round(F.sum("total_price"), 2).alias(
            "seller_order_sales"
        )
    )
)

# check the number of rows in each dataframe created:
print(f"Enriched order rows  : {enriched.count():,}")
print(f"Order rows           : {order_totals.count():,}")
print(f"Order-category rows  : {order_category.count():,}")
print(f"Order-seller rows    : {order_seller.count():,}")


Enriched order rows  : 112,650
Order rows           : 98,666
Order-category rows  : 99,470
Order-seller rows    : 100,010


# Cumulative sales per customer

Compute running total of total_price partitioned by customer_id, ordered by order_purchase_timestamp.

In [5]:
# create a window for each customer id, ordered by order_purchase_timestamp and window function with starts from the first row
# and add them 1 by 1

customer_window = (
    Window
    .partitionBy("customer_id")
    .orderBy("order_purchase_timestamp", "order_id")
    .rowsBetween(
        Window.unboundedPreceding,
        Window.currentRow,
    )
)

# create a window for each customer unique id, combining even more the orders and window function with starts from the first row
# and add them 1 by 1
person_window = (
    Window
    .partitionBy("customer_unique_id")
    .orderBy("order_purchase_timestamp", "order_id")
    .rowsBetween(
        Window.unboundedPreceding,
        Window.currentRow,
    )
)

cumulative_sales = (
    order_totals
    .withColumn("cumulative_sales", F.round(F.sum("total_price").over(customer_window), 2,))
    .withColumn("cumulative_sales_by_person", F.round( F.sum("total_price").over(person_window), 2,))
    .select(
        "order_id",
        "customer_id",
        "customer_unique_id",
        "order_purchase_timestamp",
        "total_price",
        "cumulative_sales",
        "cumulative_sales_by_person",
        "year",
        "month",
        "day",
    )
)

cumulative_sales = write_gold(cumulative_sales, "cumulative_sales_per_customer")

cumulative_sales.orderBy("customer_unique_id", "order_purchase_timestamp").show(10, truncate=False)


cumulative_sales_per_customer          rows=  98,666
+--------------------------------+--------------------------------+--------------------------------+------------------------+-----------+----------------+--------------------------+----+-----+---+
|order_id                        |customer_id                     |customer_unique_id              |order_purchase_timestamp|total_price|cumulative_sales|cumulative_sales_by_person|year|month|day|
+--------------------------------+--------------------------------+--------------------------------+------------------------+-----------+----------------+--------------------------+----+-----+---+
|e22acc9c116caa3f2b7121bbb380d08e|fadbb3709178fc513abc1b2670aa1ad2|0000366f3b9a7992bf8c76cfdf3221e2|2018-05-10 10:56:27     |141.9      |141.9           |141.9                     |2018|5    |10 |
|3594e05a005ac4d06a72673270ef9ec9|4cb282e167ae9234755102258dd52ee8|0000b849f77a49e4a4ce2b2a4ca5be3f|2018-05-07 11:11:27     |27.19      |27.19           |27.19

# Rolling average delivery time per product category

Calculate average delivery_time over the last 3 entries per product category, using a window function.

In [6]:
# the window is partitioned by product category, ordered by purchase timestamp and contains the current entry plus the previous two
# of course rows with null delivery times are excluded because they dont have a completed delivery duration.

category_window = (
    Window
    .partitionBy("product_category_name")
    .orderBy("order_purchase_timestamp", "order_id")
    .rowsBetween(-2, Window.currentRow)
)

rolling_delivery = (
    order_category
    .filter(F.col("product_category_name").isNotNull())
    .filter(F.col("delivery_time_days").isNotNull())
    .withColumn("rolling_avg_delivery_time_3",F.round( F.avg("delivery_time_days").over(category_window), 2,)) #average delivery time using the rows defined
    .withColumn("entries_in_window", F.count("*").over(category_window)) #checks the count of rows used in calculation
    .select(
        "order_id",
        "product_category_name",
        "order_purchase_timestamp",
        "delivery_time_days",
        "rolling_avg_delivery_time_3",
        "entries_in_window",
        "year",
        "month",
        "day",
    )
)

rolling_delivery = write_gold( rolling_delivery, "rolling_avg_delivery_by_category")

rolling_delivery.orderBy( "product_category_name", "order_purchase_timestamp").show(10, truncate=False)


rolling_avg_delivery_by_category       rows=  97,275
+--------------------------------+-------------------------+------------------------+------------------+---------------------------+-----------------+----+-----+---+
|order_id                        |product_category_name    |order_purchase_timestamp|delivery_time_days|rolling_avg_delivery_time_3|entries_in_window|year|month|day|
+--------------------------------+-------------------------+------------------------+------------------+---------------------------+-----------------+----+-----+---+
|616f1f539d7add607844e0199fed7ea6|agro_industria_e_comercio|2017-01-23 07:03:04     |8                 |8.0                        |1                |2017|1    |23 |
|0d1bbf582326272fa550ed829bd2e1d4|agro_industria_e_comercio|2017-01-31 17:33:09     |9                 |8.5                        |2                |2017|1    |31 |
|31cb8821ab778cd23ebb6ce6f9e2bce0|agro_industria_e_comercio|2017-02-05 19:35:10     |5                 |7.33         

# KPI summary tables

• Total sales per product category.
• Average delivery time per seller.
• Order counts per customer state.

In [7]:
# Total sales per product category
kpi_sales_by_category = (
    enriched
    .filter(F.col("product_category_name").isNotNull())
    # said per product category, however i grouped by the timestamp partitions as well, meaning its calculated daily
    .groupBy(
        "year",
        "month",
        "day",
        "product_category_name",
    )
    .agg(
        F.round(F.sum("total_price"), 2).alias("total_sales"), 
        # needed only the total sales here, but i included also the profit, the item and order count
        F.round(F.sum("profit_margin"), 2).alias("total_profit"),
        F.countDistinct("order_id").alias("order_count"),
        F.count("*").alias("item_count"),
    )
)

kpi_sales_by_category = write_gold( kpi_sales_by_category, "kpi_sales_by_category")

kpi_sales_by_category                  rows=  18,990


In [8]:
# Average delivery time per seller
# seller id was checked in silver for null values, but i added isNotNull()) just for a secondary check
kpi_delivery_by_seller = (
    order_seller
    .filter(F.col("seller_id").isNotNull())
    .filter(F.col("delivery_time_days").isNotNull())
    .groupBy(
        "year",
        "month",
        "day",
        "seller_id",
        "seller_state",
    )
    .agg(
        F.round( F.avg("delivery_time_days"), 2).alias("avg_delivery_time_days"),
        F.countDistinct("order_id").alias("order_count"),
    )
)

kpi_delivery_by_seller = write_gold( kpi_delivery_by_seller, "kpi_delivery_by_seller")


kpi_delivery_by_seller                 rows=  68,017


In [9]:
# Order counts per customer state
# same thing, its a daily kpi
kpi_orders_by_state = (
    enriched
    .filter(F.col("customer_state").isNotNull())
    .groupBy(
        "year",
        "month",
        "day",
        "customer_state",
    )
    .agg(
        F.countDistinct("order_id").alias("order_count"),
    )
)

kpi_orders_by_state = write_gold( kpi_orders_by_state, "kpi_orders_by_state")

kpi_orders_by_state                    rows=  10,701


# Check the kpis results.


In [10]:
kpi_sales_by_category.orderBy( F.desc("total_sales")).show(10, truncate=False)

kpi_delivery_by_seller.orderBy( F.desc("avg_delivery_time_days")).show(10, truncate=False)

kpi_orders_by_state.orderBy( F.desc("order_count")).show(10, truncate=False)

+----+-----+---+----------------------+-----------+------------+-----------+----------+
|year|month|day|product_category_name |total_sales|total_profit|order_count|item_count|
+----+-----+---+----------------------+-----------+------------+-----------+----------+
|2017|11   |24 |cama_mesa_banho       |21317.39   |14432.67    |157        |195       |
|2017|11   |24 |relogios_presentes    |18992.86   |16341.18    |68         |75        |
|2017|8    |29 |pcs                   |17561.05   |16438.95    |15         |15        |
|2017|11   |24 |moveis_decoracao      |14319.95   |9138.47     |89         |136       |
|2017|9    |29 |telefonia_fixa        |13664.08   |13215.92    |1          |8         |
|2017|11   |24 |beleza_saude          |13330.15   |10228.89    |82         |89        |
|2017|11   |24 |informatica_acessorios|12334.7    |9513.54     |62         |71        |
|2017|8    |30 |pcs                   |11162.48   |10437.52    |9          |9         |
|2017|4    |18 |eletroportateis 

# Reporting Queries (GOLD Layer)

Create the temporary views for each kpi needed.

In [11]:
GOLD_TABLES = [
    "cumulative_sales_per_customer",
    "rolling_avg_delivery_by_category",
    "kpi_sales_by_category",
    "kpi_delivery_by_seller",
    "kpi_orders_by_state",
]

for name in GOLD_TABLES:
    spark.read.format("delta").load(str(GOLD_DIR / name)).createOrReplaceTempView(name)

spark.sql("SHOW VIEWS").show(truncate=False)

+---------+--------------------------------+-----------+
|namespace|viewName                        |isTemporary|
+---------+--------------------------------+-----------+
|         |cumulative_sales_per_customer   |true       |
|         |kpi_delivery_by_seller          |true       |
|         |kpi_orders_by_state             |true       |
|         |kpi_sales_by_category           |true       |
|         |rolling_avg_delivery_by_category|true       |
+---------+--------------------------------+-----------+



In [12]:
# We compute here the total sales per product category

sales_by_category = spark.sql("""
    SELECT
        product_category_name,
        ROUND(SUM(total_sales), 2) AS total_sales,
        SUM(order_count) AS total_orders,
        ROUND(SUM(total_sales) / NULLIF(SUM(order_count), 0), 2) AS avg_order_value
    FROM kpi_sales_by_category
    GROUP BY product_category_name
    ORDER BY total_sales DESC
""")

sales_by_category.show(15, truncate=False)

+----------------------+-----------+------------+---------------+
|product_category_name |total_sales|total_orders|avg_order_value|
+----------------------+-----------+------------+---------------+
|beleza_saude          |1441248.07 |8836        |163.11         |
|relogios_presentes    |1305541.61 |5624        |232.14         |
|cama_mesa_banho       |1241681.72 |9417        |131.86         |
|esporte_lazer         |1156656.48 |7720        |149.83         |
|informatica_acessorios|1059272.4  |6689        |158.36         |
|moveis_decoracao      |902511.79  |6449        |139.95         |
|utilidades_domesticas |778397.77  |5884        |132.29         |
|cool_stuff            |719329.95  |3632        |198.05         |
|automotivo            |685384.32  |3897        |175.87         |
|ferramentas_jardim    |584219.21  |3518        |166.07         |
|brinquedos            |561372.55  |3886        |144.46         |
|bebes                 |480118.0   |2885        |166.42         |
|perfumari

In [13]:
sales_by_category.limit(15).toPandas()

,product_category_name,total_sales,total_orders,avg_order_value
0,beleza_saude,1441248.07,8836,163.11
1,relogios_presentes,1305541.61,5624,232.14
2,cama_mesa_banho,1241681.72,9417,131.86
3,esporte_lazer,1156656.48,7720,149.83
4,informatica_acessorios,1059272.40,6689,158.36
5,moveis_decoracao,902511.79,6449,139.95
6,utilidades_domesticas,778397.77,5884,132.29
7,cool_stuff,719329.95,3632,198.05
8,automotivo,685384.32,3897,175.87
9,ferramentas_jardim,584219.21,3518,166.07


In [14]:
# compute the average delivery time per seller

delivery_by_seller = spark.sql("""
    SELECT
        seller_id,
        seller_state,
        ROUND(SUM(avg_delivery_time_days * order_count)/ SUM(order_count),2) AS avg_delivery_time_days,
        SUM(order_count) AS delivered_order_count
    FROM kpi_delivery_by_seller
    GROUP BY seller_id, seller_state
    ORDER BY avg_delivery_time_days ASC
""")

delivery_by_seller.show(20, truncate=False)

+--------------------------------+------------+----------------------+---------------------+
|seller_id                       |seller_state|avg_delivery_time_days|delivered_order_count|
+--------------------------------+------------+----------------------+---------------------+
|139157dd4daa45c25b0807ffff348363|SP          |1.0                   |1                    |
|5e063e85d44b0f5c3e6ec3131103a57e|SP          |1.0                   |1                    |
|6561d6bf844e464b4019442692b40e02|SP          |1.0                   |1                    |
|99a25c39b28a74d1151c35c18d178292|SP          |2.0                   |1                    |
|0af977692321d895349eded183341d28|SP          |2.0                   |2                    |
|2c00c85d30361cd2ced2969cffbbffa3|MG          |2.0                   |1                    |
|7d81e74a4755b552267cd5e081563028|SP          |2.0                   |1                    |
|751e274377499a8503fd6243ad9c56f6|SP          |2.0                   |

In [15]:
delivery_by_seller.limit(20).toPandas()

,seller_id,seller_state,avg_delivery_time_days,delivered_order_count
0,139157dd4daa45c25b0807ffff348363,SP,1.0,1
1,5e063e85d44b0f5c3e6ec3131103a57e,SP,1.0,1
2,6561d6bf844e464b4019442692b40e02,SP,1.0,1
3,99a25c39b28a74d1151c35c18d178292,SP,2.0,1
4,0af977692321d895349eded183341d28,SP,2.0,2
5,2c00c85d30361cd2ced2969cffbbffa3,MG,2.0,1
6,751e274377499a8503fd6243ad9c56f6,SP,2.0,1
7,37303482a42fb700d8d127e70a9cd6c8,SP,2.0,1
8,734def04b237117a09321dd6d8f3f2a2,SP,2.0,2
9,eae9af4811c294c56795d70e715b7337,SP,2.0,1


In [16]:
# compute the mumber of orders by customer state

orders_by_state = spark.sql("""
    SELECT
        customer_state,
        SUM(order_count) AS total_orders,
        ROUND(100.0 * SUM(order_count)/ SUM(SUM(order_count)) OVER (),2) AS pct_of_orders
    FROM kpi_orders_by_state
    WHERE customer_state IS NOT NULL
    GROUP BY customer_state
    ORDER BY total_orders DESC
""")

orders_by_state.show(27, truncate=False)

+--------------+------------+-------------+
|customer_state|total_orders|pct_of_orders|
+--------------+------------+-------------+
|SP            |41375       |41.93        |
|RJ            |12762       |12.93        |
|MG            |11544       |11.70        |
|RS            |5432        |5.51         |
|PR            |4998        |5.07         |
|SC            |3612        |3.66         |
|BA            |3358        |3.40         |
|DF            |2125        |2.15         |
|ES            |2025        |2.05         |
|GO            |2007        |2.03         |
|PE            |1648        |1.67         |
|CE            |1327        |1.34         |
|PA            |970         |0.98         |
|MT            |903         |0.92         |
|MA            |740         |0.75         |
|MS            |709         |0.72         |
|PB            |532         |0.54         |
|PI            |493         |0.50         |
|RN            |482         |0.49         |
|AL            |411         |0.4

# Insights based on query outputs.

In [17]:
# this query is used to rank the hightest selling product category to the lowest, provide the total sales and the percentage it has overall
# With the cte, we retrieve the total sales per category
category_sales_rank = spark.sql("""
    WITH totals AS (
        SELECT
        product_category_name,
        SUM(total_sales) AS category_sales
        FROM kpi_sales_by_category
        GROUP BY product_category_name
    )
    SELECT
        product_category_name,
        ROUND(category_sales, 2) AS category_sales,
        ROUND(
            100.0 * category_sales
            / SUM(category_sales) OVER (),
            2
        ) AS percentage_of_sales,
        DENSE_RANK() OVER (
            ORDER BY category_sales DESC
        ) AS sales_rank
    FROM totals
    ORDER BY sales_rank
""")

category_sales_rank.show(10, truncate=False)

+----------------------+--------------+-------------------+----------+
|product_category_name |category_sales|percentage_of_sales|sales_rank|
+----------------------+--------------+-------------------+----------+
|beleza_saude          |1441248.07    |9.1                |1         |
|relogios_presentes    |1305541.61    |8.24               |2         |
|cama_mesa_banho       |1241681.72    |7.84               |3         |
|esporte_lazer         |1156656.48    |7.3                |4         |
|informatica_acessorios|1059272.4     |6.69               |5         |
|moveis_decoracao      |902511.79     |5.7                |6         |
|utilidades_domesticas |778397.77     |4.91               |7         |
|cool_stuff            |719329.95     |4.54               |8         |
|automotivo            |685384.32     |4.33               |9         |
|ferramentas_jardim    |584219.21     |3.69               |10        |
+----------------------+--------------+-------------------+----------+
only s

In [18]:
top_categories = category_sales_rank.limit(3).collect() # collect the first 3 rows and convert them into a python list of row objects

print("Insights regarding product category:")
if top_categories:
    for row in top_categories:
        print(
            f"Rank {row['sales_rank']}: "
            f"{row['product_category_name']} generated "
            f"{row['category_sales']:,.2f}, making up "
            f"{row['percentage_of_sales']:.2f}% of total sales."
        )
else:
    print("No category results were returned.")

Insights regarding product category:
Rank 1: beleza_saude generated 1,441,248.07, making up 9.10% of total sales.
Rank 2: relogios_presentes generated 1,305,541.61, making up 8.24% of total sales.
Rank 3: cama_mesa_banho generated 1,241,681.72, making up 7.84% of total sales.


In [19]:
top_states = orders_by_state.limit(5).collect() # collect the first 5 rows and convert them into a python list of row objects

print("Insights regarding customer state: ")
if top_states:
    position = 1
    for row in top_states:
        print(
            f"{position}. {row['customer_state']}: "
            f"{row['total_orders']:,} orders, "
            f"{row['pct_of_orders']:.2f}% of all orders."
        )
        position += 1
else:
    print("No state results were returned.")

Insights regarding customer state: 
1. SP: 41,375 orders, 41.93% of all orders.
2. RJ: 12,762 orders, 12.93% of all orders.
3. MG: 11,544 orders, 11.70% of all orders.
4. RS: 5,432 orders, 5.51% of all orders.
5. PR: 4,998 orders, 5.07% of all orders.


In [20]:
# for the delivery time for sellers, we will check the fastest and the slowest between them, starting from a threshold
# of 20 orders. 

delivery_by_seller_20 = (
    delivery_by_seller
    .filter(F.col("delivered_order_count") >= 20)
)

fastest = (
    delivery_by_seller_20
    .orderBy(F.asc("avg_delivery_time_days"))
    .limit(1)
    .collect()
)

slowest = (
    delivery_by_seller_20
    .orderBy(F.desc("avg_delivery_time_days"))
    .limit(1)
    .collect()
)

print("Insights regarding seller delivery: ")

if fastest:
    row = fastest[0]
    print(
        f"Fastest seller with at least 20 orders: "
        f"{row['seller_id']} at "
        f"{row['avg_delivery_time_days']} days."
    )

if slowest:
    row = slowest[0]
    print(
        f"Slowest seller with at least 20 orders: "
        f"{row['seller_id']} at "
        f"{row['avg_delivery_time_days']} days."
    )

Insights regarding seller delivery: 
Fastest seller with at least 20 orders: 41c2bad7229b0c25e6becf179ebf63ff at 4.5 days.
Slowest seller with at least 20 orders: 66e0557ecc2b4dbea057e93f215f68d8 at 31.63 days.


In [21]:
spark.stop()